In [1]:

import time
from typing import Any

import gymnasium as gym
import numpy as np
import torch
from torch import optim, nn

from src.experiment_logging.experiment_log import ExperimentLogItem
from src.experiment_logging.experiment_logger import ExperimentLogger, log_experiment
from src.module_analysis import count_parameters
from src.reinforcement_learning.algorithms.sac.sac import SAC, SACInfoStashConfig
from src.reinforcement_learning.algorithms.sac.sac_policy import SACPolicy
from src.reinforcement_learning.core.action_selectors.predicted_std_action_selector import PredictedStdActionSelector
from src.reinforcement_learning.core.callback import Callback
from src.reinforcement_learning.core.loss_config import LossInfoStashConfig
from src.reinforcement_learning.core.policies.components.actor import Actor
from src.reinforcement_learning.core.policies.components.q_critic import QCritic
from src.reinforcement_learning.gym.parallelize_env import parallelize_env_async
from src.stopwatch import Stopwatch
from src.summary_statistics import maybe_compute_summary_statistics
from swarmbots.swarm_bot_env import SwarmBotEnv

%load_ext autoreload
%autoreload 2

pygame 2.5.2 (SDL 2.28.3, Python 3.11.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
def get_setup() -> dict[str, str]:
    return {
        'notebook': _ih[1] + '\n\n' + _ih[-1], # first and last cell input (imports and this cell)
    }

step_stopwatch = Stopwatch()
total_stopwatch = Stopwatch()
best_iteration_score = -1e6

save_best = True

def save_policy(suffix: str, score: float, steps_performed: int, gradient_steps_performed: int):
    experiment_id = logger.experiment_log["experiment_id"]
    policy.save(
        f'saved_models/{experiment_id}_{suffix}.pth', 
        score=score, 
        steps_performed=steps_performed,
        gradient_steps_performed=gradient_steps_performed
    )

def on_rollout_done(rl: SAC, step: int, info: dict[str, Any], scheduler_values: dict[str, Any]):
    if step % 1000 != 0:
        return
    
    episode_scores = rl.buffer.compute_most_recent_episode_scores(2, consider_truncated_as_done=True)
    
    if save_best and len(episode_scores) > 0:
    
        global best_iteration_score
        iteration_score = episode_scores.mean()
        if iteration_score >= best_iteration_score:
            save_policy('best', iteration_score, step, rl.gradient_steps_performed)
            best_iteration_score = iteration_score
    
    info['episode_scores'] = episode_scores
        
def on_optimization_done(rl: SAC, step: int, info: dict[str, Any], scheduler_values: dict[str, Any]):    
    if step % 1000 != 0:
        return
    
    num_env_steps = step * rl.num_envs
    
    step_time = step_stopwatch.reset()
    total_time = total_stopwatch.time_passed()
    
    tail_indices = rl.buffer.tail_indices(1000)
    
    episode_scores = info.get('episode_scores')
    
    log_item: ExperimentLogItem = {
        'step': step,
        'num_env_steps': num_env_steps,
        'scores': maybe_compute_summary_statistics(episode_scores),
        'actor_loss': maybe_compute_summary_statistics(info['final_actor_loss']),
        'critic_loss': maybe_compute_summary_statistics(info['final_critic_loss']),
        'entropy_coef_loss': maybe_compute_summary_statistics(info.get('final_entropy_coef_loss')),
        'entropy_coef': maybe_compute_summary_statistics(info['entropy_coef']),
        'action_stds': maybe_compute_summary_statistics(info['rollout'].get('action_stds')),
        'action_magnitude': maybe_compute_summary_statistics(np.abs(rl.buffer.actions[tail_indices])),
        'num_gradient_steps': rl.gradient_steps_performed,
        'step_time': step_time,
        'total_time': total_time
    }
    print(logger.format_log_item(
        log_item, 
        mean_format='5.3f', 
        std_format='5.3f', 
        step_time='.2f', 
        total_time='.2f',
        scores={
            'mean_format': '5.3f',
            'n_format': 'd',
        }
    ), end='\n\n')
    logger.add_item(log_item)
    if step % 10000 == 0:
        logger.save_experiment_log()
        
        if step % 100_000 == 0:
            save_policy('latest', episode_scores.mean(), step, rl.gradient_steps_performed)
            
        print()
    print()

device = torch.device("cuda:0") if True else torch.device('cpu')
print(f'using device {device}')

env_kwargs = {}
num_envs = 1

def create_env(render_mode: str | None):
    make_single_env = lambda: SwarmBotEnv(
        physics_steps_per_step=10, 
        action_scale=5, 
        hip_range=np.pi / 4,
        ctrl_cost_weight=0.001,
    )
    
    if num_envs == 1:
        return make_single_env()
        
    return parallelize_env_async(make_single_env, num_envs)


def create_policy():
    in_size = int(np.prod(env.observation_space.shape))
    latent_action_size = 256
    action_size = int(np.prod(env.action_space.shape))
    
    actor_net = nn.Sequential(
        nn.Linear(in_size, 256),
        nn.ReLU(),
        nn.Linear(256, latent_action_size),
        nn.ReLU(),
    )

    critic = QCritic(
        n_critics=2,
        create_q_network=lambda: nn.Sequential(
            nn.Linear(in_size + action_size, 256),
            nn.ReLU(),
            # BatchRenorm(256),
            nn.Linear(256, 256),
            nn.ReLU(),
            # BatchRenorm(256),
            nn.Linear(256, 1)
        )
    )

    return SACPolicy(
        actor=Actor(actor_net, PredictedStdActionSelector(
            latent_dim=latent_action_size,
            action_dim=action_size,
            base_std=1.0,
            squash_output=True,
        )),
        critic=critic
    )


env = create_env(render_mode=None)
policy = create_policy()
logger = ExperimentLogger(f'experiment_logs/')

try:
    print(f'{count_parameters(policy) = }')
    print(f'{env = }, {num_envs = }')
        
    with ((torch.autograd.set_detect_anomaly(False))):
        algo = SAC(
            env=env,
            policy=policy,
            actor_optimizer_provider=lambda params: optim.Adam(params, lr=3e-4),  # (params, lr=3e-4, betas=(0.5, 0.999)),
            critic_optimizer_provider=lambda params: optim.Adam(params, lr=3e-4),  # (params, lr=3e-4, betas=(0.5, 0.999)),
            buffer_size=1_000_000,
            reward_scale=10,
            gamma=0.99,
            tau=0.005,
            entropy_coef_optimizer_provider=lambda params: optim.Adam(params, lr=3e-4),
            entropy_coef_clamp_range=(0.001, 1.5),
            # entropy_coef=0.1,
            rollout_steps=1000,
            gradient_steps=1000,
            warmup_steps=10_000,
            optimization_batch_size=256,
            target_update_interval=1,
            callback=Callback(
                on_rollout_done=on_rollout_done,
                rollout_schedulers={},
                on_optimization_done=on_optimization_done,
                optimization_schedulers={},
            ),
            stash_config=SACInfoStashConfig(stash_rollout_infos=True, stash_rollout_action_stds=True,
                                            stash_entropy_coef=True,
                                            entropy_coef_loss=LossInfoStashConfig(stash_final=True),
                                            actor_loss=LossInfoStashConfig(stash_final=True),
                                            critic_loss=LossInfoStashConfig(stash_final=True)),
            torch_device=device,
        )
        # algo.load('saved_models/2024-10-28_11-12-19_363042~VskSv0/', '2024-10-28_11-12-19_363042~VskSv0')
        total_stopwatch.reset()
        with log_experiment(
            logger,
            # experiment_id='2024-10-28_11-12-19_363042~VskSv0_1',
            experiment_tags=algo.collect_tags(),
            hyper_parameters=algo.collect_hyper_parameters(),
            setup=get_setup(),
        ) as x:
            logger.save_experiment_log()
            print('\nStarting Training\n\n')
            # import cProfile
            # pr = cProfile.Profile()
            # pr.enable()
            algo.learn(5_000_000)
            # pr.disable()  
            # pr.dump_stats('profile_stats.pstat')
except KeyboardInterrupt as ki:
    print('keyboard interrupt')
    # raise ki
finally:
    print('closing envs')
    time.sleep(0.5)
    env.close()
    print('envs closed')
    

print('done')

using device cuda:0
count_parameters(policy) = 259618
env = <swarmbots.swarm_bot_env.SwarmBotEnv object at 0x000002A2252DC190>, num_envs = 1
Grabbing system information... done!
saved experiment log 2024-10-29_13-21-44_070699~YbqBEq at experiment_logs/2024-10-29_13-21-44_070699~YbqBEq.json

Starting Training

step = 11000, num_env_steps = 11000, scores = -4.056 ± 0.05 (n=2), actor_loss = -33.015 ± 9.285, critic_loss = 20.728 ± 8.699, entropy_coef_loss = -4.111 ± 2.339, entropy_coef = 0.861 ± 0.076, action_stds = 1.232 ± 1.088, action_magnitude = 0.648 ± 0.305, num_gradient_steps = 1000, step_time = 16.09, total_time = 16.04

step = 12000, num_env_steps = 12000, scores = -3.306 ± 0.19 (n=2), actor_loss = -60.692 ± 6.985, critic_loss = 11.915 ± 2.195, entropy_coef_loss = -11.663 ± 2.155, entropy_coef = 0.639 ± 0.054, action_stds = 0.850 ± 0.081, action_magnitude = 0.525 ± 0.287, num_gradient_steps = 2000, step_time = 9.04, total_time = 25.07

step = 13000, num_env_steps = 13000, scores =

In [4]:
from src.reinforcement_learning.core.policy_evaluation import record_policy

env.render_mode = 'rgb_array'
record_policy(
    env=env,
    policy=policy,
    video_folder='videos',
    deterministic_actions=False,
    num_steps=999,
    torch_device=device,
)

C:\Program Files\Python311\Lib\site-packages\gymnasium\wrappers\record_video.py:94: UserWarning: WARN: Overwriting existing videos at C:\Users\domin\Git\swarm-bots\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Moviepy - Building video C:\Users\domin\Git\swarm-bots\videos\rl-video-episode-0.mp4.
Moviepy - Writing video C:\Users\domin\Git\swarm-bots\videos\rl-video-episode-0.mp4


Moviepy - Done !
Moviepy - video ready C:\Users\domin\Git\swarm-bots\videos\rl-video-episode-0.mp4
closing record env
Moviepy - Building video C:\Users\domin\Git\swarm-bots\videos\rl-video-episode-1.mp4.
Moviepy - Writing video C:\Users\domin\Git\swarm-bots\videos\rl-video-episode-1.mp4


Moviepy - Done !
Moviepy - video ready C:\Users\domin\Git\swarm-bots\videos\rl-video-episode-1.mp4
record env closed


In [10]:
env.physics.data.xpos

array([[ 0.        ,  0.        ,  0.        ],
       [10.55573153, -0.4078065 ,  0.14921404],
       [10.55573153, -0.4078065 ,  0.14921404],
       [10.48739983, -0.43612304,  0.21651153],
       [10.48739983, -0.43612304,  0.21651153],
       [10.48739983, -0.43612304,  0.21651153],
       [10.64728733, -0.4268981 ,  0.18461243],
       [10.64728733, -0.4268981 ,  0.18461243],
       [10.64728733, -0.4268981 ,  0.18461243],
       [10.54656564, -0.30979478,  0.13161603],
       [10.54656564, -0.30979478,  0.13161603],
       [10.54656564, -0.30979478,  0.13161603],
       [10.02058614, -0.30683126,  0.33239675],
       [10.02058614, -0.30683126,  0.33239675],
       [10.11026708, -0.30289121,  0.37646266],
       [10.11026708, -0.30289121,  0.37646266],
       [10.11026708, -0.30289121,  0.37646266],
       [10.00786531, -0.22566631,  0.27538418],
       [10.00786531, -0.22566631,  0.27538418],
       [10.00786531, -0.22566631,  0.27538418],
       [ 9.94918638, -0.30995894,  0.402

In [2]:
import PIL.Image

PIL.Image.fromarray(create_env(None).render())

NameError: name 'create_env' is not defined